# 행동 배열 생성 코드

팀에서 제공한 원본 생성 코드와 실행 기록입니다. 입력 자료가 관찰 기간으로 제한됐다는 전제로 검토했습니다.
생성 규칙을 확인했으며 이번 검토에서 입력 원본의 날짜 범위를 직접 재검사하거나 현재 저장 배열 전체를 재생성해 비교하지 않았습니다.
노트북의 입력 경로는 원래 작업 환경 기준입니다. 파케 입력을 읽으려면 별도로 폴라스 패키지가 필요합니다.

In [1]:
import json

def extract_user_ids(file_path):
    user_ids = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # 빈 줄이 있는 경우 건너뜁니다
            if line.strip():
                data = json.loads(line)
                if 'user_id' in data:
                    user_ids.append(data['user_id'])
    return user_ids

In [2]:
file_path = './데이터/customer_sequences_20181101_v1.jsonl'
user_id_list = extract_user_ids(file_path)
print(user_id_list)

['1000035', '1000103', '1000164', '1000168', '1000182', '1000194', '1000214', '1000233', '1000244', '1000248', '1000280', '1000407', '1000409', '1000446', '1000527', '1000611', '1000625', '1000626', '1000655', '1000662', '1000675', '1000885', '1000963', '1001027', '1001129', '1001177', '1001185', '1001197', '1001201', '1001246', '1001256', '1001367', '1001393', '1001559', '1001607', '1001725', '1001728', '1001759', '1001862', '1001872', '1001880', '1001904', '1001958', '1001962', '1001963', '1002143', '1002185', '1002221', '1002283', '1002295', '1002324', '1002397', '1002404', '1002409', '1002476', '1002491', '1002530', '1002533', '1002614', '1002712', '1002748', '1002749', '1002751', '1002755', '1002763', '1002879', '1003092', '1003173', '1003214', '1003452', '1003520', '1003616', '1003633', '1003703', '1003847', '1003884', '1003888', '1003912', '1003939', '1004001', '1004060', '1004284', '1004304', '1004316', '1004479', '1004498', '1004554', '1004720', '1004747', '1004753', '1004765'

In [3]:
import polars as pl
import numpy as np
import pandas as pd

# Parquet 파일 읽기
df = pl.read_parquet("./데이터/processed_data.parquet")

print(f"원래 데이터 전체 행 수: {df.height:,} 건, 유저 수: {df['userId'].n_unique():,} 명")

# 🚨 [핵심 수정] LSTM 모델의 데이터 누수(AUC 1.0)를 유발하는 치트키 페이지 제거
# 이탈을 직접적으로 유출하는 결정적 페이지들을 로그 시퀀스에서 제외합니다.
leakage_pages = [
    'Cancel', 
    'Cancellation Confirmation', 
    'Submit Downgrade',
    'Downgrade',          # 추가
    'Submit Upgrade',     # 추가
    'Upgrade'             # 추가
]
df = df.filter(~pl.col("page").is_in(leakage_pages))

원래 데이터 전체 행 수: 26,259,199 건, 유저 수: 22,278 명


In [4]:
# 1. 💡 [핵심 수정] 전체 데이터를 시간순(ts)으로 먼저 정렬한 뒤 유저별로 그룹화합니다.
# 이렇게 하면 .last()를 사용할 때 가장 마지막(최근)에 발생한 데이터가 정확히 선택됩니다.
user_tabular_features = (
    df.sort("ts")
    .group_by("userId")
    .agg([
        pl.col("is_churn").first().alias("is_churn"),
        pl.col("gender").first().alias("gender"),
        pl.col("level").last().alias("level"),
        
        # 원본 타임스탬프 및 세션 관련 (나중에 tenure_days 계산 등에 활용)
        pl.col("ts").max().alias("max_ts"),
        pl.col("registration").first().alias("registration"),
        
        # 기본 누적 통계
        pl.len().alias("total_interactions"),
        (pl.col("page") == "NextSong").sum().alias("total_songs_played"),
        (pl.col("page") == "Thumbs Up").sum().alias("total_thumbs_up"),
        (pl.col("page") == "Thumbs Down").sum().alias("total_thumbs_down"),
        (pl.col("page") == "Add to Playlist").sum().alias("total_playlist_adds"),
        
        # 서비스 불편도 및 광고 피로도 피처
        (pl.col("page") == "Error").sum().alias("total_errors"),
        (pl.col("page") == "Help").sum().alias("total_help"),
        
        pl.col("sessionId").n_unique().alias("total_sessions"),
        pl.col("length").sum().alias("total_listening_time_sec"),
    ])
)

In [5]:
# 유저별로 최근 최대 N개의 이벤트 시퀀스 추출 (예: 최근 50개 행동)
MAX_SEQ_LEN = 50

user_sequences = (
    df.sort(["userId", "ts"])
    .group_by("userId")
    .agg([
        pl.col("page").tail(MAX_SEQ_LEN).alias("page_sequence"),
        pl.col("itemInSession").tail(MAX_SEQ_LEN).alias("session_item_sequence")
    ])
)

# 정형 피처와 시퀀스 피처를 유저 ID 기준 병합
user_master_df = user_tabular_features.join(user_sequences, on="userId")

print(f"최종 마스터 유저 테이블 크기: {user_master_df.shape}")
print(user_master_df.select(["userId", "is_churn", "page_sequence"]).head(3))

최종 마스터 유저 테이블 크기: (22278, 17)
shape: (3, 3)
┌─────────┬──────────┬─────────────────────────────────┐
│ userId  ┆ is_churn ┆ page_sequence                   │
│ ---     ┆ ---      ┆ ---                             │
│ str     ┆ i32      ┆ list[str]                       │
╞═════════╪══════════╪═════════════════════════════════╡
│ 1000025 ┆ 1        ┆ ["NextSong", "Home", … "NextSo… │
│ 1000035 ┆ 0        ┆ ["Thumbs Up", "NextSong", … "T… │
│ 1000083 ┆ 1        ┆ ["NextSong", "NextSong", … "Ne… │
└─────────┴──────────┴─────────────────────────────────┘


In [20]:
import polars as pl
import numpy as np
import ast

# 2, 3, 4번 파일과 각각 저장될 train, valid, test.npy 파일 매핑
split_files = {
    "./데이터/train_20181101_v1.csv": "train.npy",  
    "./데이터/valid_20181101_v1.csv": "valid.npy",
    "./데이터/test_20181101_v1.csv": "test.npy"
}

# 설정: 모델 학습에 필요한 최대 시퀀스 길이 및 패딩 값
MAX_LEN = 50        
PADDING_VALUE = 0   # 0은 패딩(빈자리)용으로 예약합니다.

# 1. 문자열 페이지 이름을 숫자 ID로 바꾸기 위한 맵핑(Dictionary) 생성
# user_master_df에 있는 모든 page_sequence에서 중복 없는 고유 페이지 목록 추출
all_sequences = user_master_df["page_sequence"].to_list()
unique_pages = set()
for seq in all_sequences:
    if seq is not None:
        if isinstance(seq, str):
            seq = ast.literal_eval(seq)
        unique_pages.update(seq)

# 0번은 패딩(PADDING_VALUE)에 양보하고, 1부터 고유 페이지별로 숫자 ID 부여
page_to_idx = {page: idx + 1 for idx, page in enumerate(sorted(list(unique_pages)))}
print(f"고유 페이지 종류 개수: {len(page_to_idx)}")

for input_path, output_path in split_files.items():
    # 2. 파일 읽기 (CSV 또는 JSON)
    if input_path.endswith('.json'):
        split_df = pl.read_json(input_path)
    else:
        split_df = pl.read_csv(input_path)
    
    # 3. 타입 불일치 해결을 위해 split_df의 user_id를 String 타입으로 캐스팅 후 join
    merged = split_df.select(
        pl.col("user_id").cast(pl.String)
    ).join(
        user_master_df, 
        left_on="user_id", 
        right_on="userId", 
        how="left"
    )
    
    # 4. page_sequence 컬럼 추출
    sequences = merged["page_sequence"].to_list()
    
    # 5. 고정 길이의 2D numpy 배열 생성 (패딩 적용)
    padded_sequences = np.full((len(sequences), MAX_LEN), PADDING_VALUE, dtype=np.int32)
    
    for i, seq in enumerate(sequences):
        if seq is not None:
            # 문자열 형태인 경우 리스트로 파싱
            if isinstance(seq, str):
                seq = ast.literal_eval(seq)
            seq = list(seq)
            
            # 문자열 페이지 이름을 숫자 ID로 변환 (사전에 없는 값이면 PADDING_VALUE 처리)
            encoded_seq = [page_to_idx.get(page, PADDING_VALUE) for page in seq]
            
            seq_len = len(encoded_seq)
            if seq_len > MAX_LEN:
                # 길이를 초과하는 경우 최근 기록(뒤쪽)을 유지
                padded_sequences[i] = encoded_seq[-MAX_LEN:]
            else:
                # 짧은 경우 앞쪽에 채우기
                padded_sequences[i, :seq_len] = encoded_seq
        # seq가 None이거나 비어있는 경우 PADDING_VALUE 유지
        
    # 6. .npy 파일로 최종 저장
    np.save(output_path, padded_sequences)
    print(f"'{input_path}' 처리 완료 -> '{output_path}' 저장됨 | 최종 Shape: {padded_sequences.shape}")

고유 페이지 종류 개수: 14
'./데이터/train_20181101_v1.csv' 처리 완료 -> 'train.npy' 저장됨 | 최종 Shape: (12697, 50)
'./데이터/valid_20181101_v1.csv' 처리 완료 -> 'valid.npy' 저장됨 | 최종 Shape: (2721, 50)
'./데이터/test_20181101_v1.csv' 처리 완료 -> 'test.npy' 저장됨 | 최종 Shape: (2721, 50)
